# 8 Click Prediction Hyperparameter Tuning v1

## 8.0 Purpose

This notebook tunes the Stage 6 shortlisted click-prediction candidates using only the train-validation pool.

Shortlisted candidates:
1. Logistic Regression + C_expanded
2. Random Forest + C_expanded

Primary tuning metric:
- PR-AUC

Supporting metrics:
- ROC-AUC
- Brier score
- Log loss
- Accuracy
- Balanced accuracy

Fixed rules:
- The final chronological test set is not used during tuning.
- Hyperparameters are selected using grouped cross-validation only.
- `mailing_id` is used as the grouping variable.
- Preprocessing remains inside sklearn Pipeline / ColumnTransformer.
- Train and validation scores are saved to diagnose overfitting and underfitting.
- If two settings perform similarly, the simpler or more stable setting is preferred.

## 8.1 Imports and paths

In [1]:
# imports
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedGroupKFold
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    balanced_accuracy_score
)

In [3]:
# project paths
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "Data" / "Processed Data"
OUTPUT_DIR = PROJECT_ROOT / "Outputs"

CLICK_DATA_PATH = DATA_DIR / "df_click_model_v1.parquet"

SHORTLIST_PATH = OUTPUT_DIR / "6_click_shortlist_v1.csv"

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)

print("\nClick data exists:", CLICK_DATA_PATH.exists())
print("Shortlist exists:", SHORTLIST_PATH.exists())

Project root: /Users/khanhhien/Documents/Thesis
Data dir: /Users/khanhhien/Documents/Thesis/Data/Processed Data
Output dir: /Users/khanhhien/Documents/Thesis/Outputs

Click data exists: True
Shortlist exists: True


In [4]:
# reproducibility
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

print("Random state =", RANDOM_STATE)

Random state = 42


In [5]:
NOTEBOOK_NAME = "8_click_hyperparameter_tuning_v1"

TARGET_NAME = "click"

PRIMARY_METRIC = "pr_auc"

print(f"Notebook: {NOTEBOOK_NAME}")
print(f"Target: {TARGET_NAME}")
print(f"Primary metric: {PRIMARY_METRIC}")

Notebook: 8_click_hyperparameter_tuning_v1
Target: click
Primary metric: pr_auc


## 8.2 Load Stage 6 shortlist

In [6]:
# Stage 6 shortlist (frozen)
CLICK_SHORTLIST = [
    "logistic_regression__C_expanded",
    "random_forest__C_expanded"
]

print("Click shortlist:")
for model in CLICK_SHORTLIST:
    print("-", model)

Click shortlist:
- logistic_regression__C_expanded
- random_forest__C_expanded


## 8.3 Load click modelling dataset

In [ ]:
# load click modelling dataset
df_click = pd.read_parquet(CLICK_DATA_PATH)

print("Click dataset loaded.")
print("Shape:", df_click.shape)
print("Columns:", df_click.shape[1])

Click dataset loaded.
Shape: (1104242, 38)
Columns: 38


In [9]:
# basic target and campaign checks
TARGET_COL = "click"
GROUP_COL = "mailing_id"

required_cols = [TARGET_COL, GROUP_COL, "user_id"]

missing_required_cols = [col for col in required_cols if col not in df_click.columns]

print("Missing required columns:", missing_required_cols)

print("\nTarget distribution:")
print(df_click[TARGET_COL].value_counts(dropna=False))

print("\nTarget rate:")
print(df_click[TARGET_COL].mean())

print("\nUnique campaigns:", df_click[GROUP_COL].nunique())
print("Campaign ID range:", df_click[GROUP_COL].min(), "to", df_click[GROUP_COL].max())

assert len(missing_required_cols) == 0, "Some required columns are missing."
assert df_click[TARGET_COL].isna().sum() == 0, "Target contains missing values."

Missing required columns: []

Target distribution:
click
0.0    1103047
1.0       1195
Name: count, dtype: int64

Target rate:
0.001082190316977619

Unique campaigns: 274
Campaign ID range: 653 to 1418


## 8.4 Recreate chronological campaign split

In [10]:
# chronological campaign holdout
FINAL_TEST_SIZE = 0.20
TRAINVAL_SIZE = 1 - FINAL_TEST_SIZE

campaign_ids = np.sort(df_click[GROUP_COL].dropna().unique())

n_campaigns = len(campaign_ids)
n_trainval_campaigns = int(np.floor(n_campaigns * TRAINVAL_SIZE))

trainval_campaigns = campaign_ids[:n_trainval_campaigns]
test_campaigns = campaign_ids[n_trainval_campaigns:]

trainval_df = df_click[df_click[GROUP_COL].isin(trainval_campaigns)].copy()
test_df = df_click[df_click[GROUP_COL].isin(test_campaigns)].copy()

print("Total campaigns:", n_campaigns)
print("Train-validation campaigns:", len(trainval_campaigns))
print("Final test campaigns:", len(test_campaigns))

print("\nTrain-validation shape:", trainval_df.shape)
print("Final test shape:", test_df.shape)

print("\nTrain-validation campaign range:", trainval_campaigns.min(), "to", trainval_campaigns.max())
print("Final test campaign range:", test_campaigns.min(), "to", test_campaigns.max())

Total campaigns: 274
Train-validation campaigns: 219
Final test campaigns: 55

Train-validation shape: (814269, 38)
Final test shape: (289973, 38)

Train-validation campaign range: 653 to 1252
Final test campaign range: 1257 to 1418


In [11]:
# split quality checks
campaign_overlap = set(trainval_campaigns).intersection(set(test_campaigns))

print("Campaign overlap:", len(campaign_overlap))

print("\nClick rate:")
print("Train-validation:", trainval_df[TARGET_COL].mean())
print("Final test:", test_df[TARGET_COL].mean())

print("\nRows:")
print("Train-validation rows:", len(trainval_df))
print("Final test rows:", len(test_df))

assert len(campaign_overlap) == 0, "Campaign leakage: trainval and test share mailing_id."
assert len(trainval_df) + len(test_df) == len(df_click), "Split row counts do not sum to full dataset."

Campaign overlap: 0

Click rate:
Train-validation: 0.001228095383712262
Final test: 0.0006724764029754494

Rows:
Train-validation rows: 814269
Final test rows: 289973


## 8.5 Feature set C definitions

In [12]:
click_feature_types_C = {
    "numeric": [
        "age_clean",
        "n_interests",
        "subject_length",
        "preheader_length",
        "prior_email_count",
        "prior_open_count",
        "prior_click_count",
        "historical_open_rate",
        "historical_click_rate",
    ],
    
    "categorical": [
        "gender",
        "age_group",
        "campaign_topic",
        "subject_length_group",
        "preheader_length_group",
        "historical_open_bucket",
        "prior_email_frequency_bucket",
    ],
    
    "binary": [
        "interest_topic_match",
        "has_campaign_metadata",
        "subject_missing",
        "subject_contains_number",
        "subject_contains_promotion",
        "subject_contains_exclamation",
        "preheader_missing",
        "preheader_contains_number",
        "preheader_contains_promotion",
        "preheader_contains_exclamation",
        "had_prior_click",
        "is_first_email",
    ],
}

In [13]:
all_features_C = (
    click_feature_types_C["numeric"]
    + click_feature_types_C["categorical"]
    + click_feature_types_C["binary"]
)

print("Total features:", len(all_features_C))

missing_features = [
    col for col in all_features_C
    if col not in trainval_df.columns
]

print("Missing features:", missing_features)

Total features: 28
Missing features: []


## 8.6 Build preprocessing pipeline

In [14]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [15]:
# build preprocessing pipeline for Feature Set C
def build_preprocessor(feature_types):
    """
    Build a ColumnTransformer for numeric, categorical, and binary features.

    Important:
    - This preprocessor is not fitted here.
    - It will be fitted only inside each CV training fold through sklearn Pipeline.
    """

    numeric_features = feature_types["numeric"]
    categorical_features = feature_types["categorical"]
    binary_features = feature_types["binary"]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    binary_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features),
            ("categorical", categorical_transformer, categorical_features),
            ("binary", binary_transformer, binary_features),
        ],
        remainder="drop"
    )

    return preprocessor

In [16]:
click_preprocessor_C = build_preprocessor(click_feature_types_C)

click_preprocessor_C

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=0,
                                                                strategy='constant')),
                                                 ('scaler', StandardScaler())]),
                                 ['age_clean', 'n_interests', 'subject_length',
                                  'preheader_length', 'prior_email_count',
                                  'prior_open_count', 'prior_click_count',
                                  'historical_open_rate',
                                  'historical_click_rate']),
                                ('categorical',
                                 Pipeline(steps=...
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent'))]),
                                 ['interest_topic_match',
                                  'has_campaign_metadata', 'subject_missing',
                                  'subject_contains_number',
                                  'subject_contains_promotion',
                                  'subject_contains_exclamation',
                                  'preheader_missing',
                                  'preheader_contains_number',
                                  'preheader_contains_promotion',
                                  'preheader_contains_exclamation',
                                  'had_prior_click', 'is_first_email'])])

## 8.7 Create modelling matrices

In [17]:
# feature matrix, target, and grouping variable
all_features_C = (
    click_feature_types_C["numeric"]
    + click_feature_types_C["categorical"]
    + click_feature_types_C["binary"]
)

X_trainval = trainval_df[all_features_C].copy()

y_trainval = trainval_df[TARGET_COL].copy()

groups_trainval = trainval_df[GROUP_COL].copy()

print("X shape:", X_trainval.shape)
print("y shape:", y_trainval.shape)
print("groups shape:", groups_trainval.shape)

print("\nTarget rate:")
print(y_trainval.mean())

print("\nUnique campaigns:")
print(groups_trainval.nunique())

X shape: (814269, 28)
y shape: (814269,)
groups shape: (814269,)

Target rate:
0.001228095383712262

Unique campaigns:
219


In [18]:
# sanity checks
assert len(X_trainval) == len(y_trainval)
assert len(X_trainval) == len(groups_trainval)

assert X_trainval.index.equals(y_trainval.index)
assert X_trainval.index.equals(groups_trainval.index)

print("All sanity checks passed.")

All sanity checks passed.


## 8.8 StratifiedGroupKFold

In [19]:
# grouped cross-validation
cv_strategy = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print(cv_strategy)

StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)


In [21]:
# fold diagnostics
fold_summary = []

for fold_idx, (train_idx, val_idx) in enumerate(
    cv_strategy.split(
        X_trainval,
        y_trainval,
        groups=groups_trainval
    ),
    start=1
):

    y_train_fold = y_trainval.iloc[train_idx]
    y_val_fold = y_trainval.iloc[val_idx]

    groups_train_fold = set(groups_trainval.iloc[train_idx])
    groups_val_fold = set(groups_trainval.iloc[val_idx])

    overlap = groups_train_fold.intersection(groups_val_fold)

    fold_summary.append({
        "fold": fold_idx,
        "train_rows": len(train_idx),
        "val_rows": len(val_idx),
        "train_click_rate": y_train_fold.mean(),
        "val_click_rate": y_val_fold.mean(),
        "group_overlap": len(overlap)
    })

fold_summary_df = pd.DataFrame(fold_summary)

fold_summary_df

,fold,train_rows,val_rows,train_click_rate,val_click_rate,group_overlap
0,1,650238,164031,0.001143,0.001567,0
1,2,650469,163800,0.001314,0.000885,0
2,3,633122,181147,0.001395,0.000646,0
3,4,666325,147944,0.001108,0.001771,0
4,5,656922,157347,0.001189,0.001392,0


In [22]:
print(
    fold_summary_df[
        ["fold",
         "train_click_rate",
         "val_click_rate",
         "group_overlap"]
    ]
)

   fold  train_click_rate  val_click_rate  group_overlap
0     1          0.001143        0.001567              0
1     2          0.001314        0.000885              0
2     3          0.001395        0.000646              0
3     4          0.001108        0.001771              0
4     5          0.001189        0.001392              0


## 8.9 Define tuning metrics

In [52]:
scoring_click = {
    "ROC_AUC": "roc_auc",
    "PR_AUC": "average_precision",
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "neg_brier": "neg_brier_score",
    "neg_log_loss": "neg_log_loss",
}

click_refit_metric = "PR_AUC"

print("Primary scoring metric:", click_refit_metric)
print("All scoring metrics:")
for metric in scoring_click:
    print("-", metric)

Primary scoring metric: PR_AUC
All scoring metrics:
- ROC_AUC
- PR_AUC
- accuracy
- balanced_accuracy
- neg_brier
- neg_log_loss


In [ ]:
# output file names
CLICK_TUNING_RESULTS_PATH = OUTPUT_DIR / "8_click_tuning_results_v1.csv"
CLICK_BEST_PARAMS_PATH = OUTPUT_DIR / "8_click_best_params_v1.json"
CLICK_TUNED_SUMMARY_PATH = OUTPUT_DIR / "8_click_tuned_model_summary_v1.csv"
CLICK_TUNING_NOTES_PATH = OUTPUT_DIR / "8_click_tuning_notes_v1.md"

print("Tuning results path:", CLICK_TUNING_RESULTS_PATH)
print("Best params path:", CLICK_BEST_PARAMS_PATH)
print("Summary path:", CLICK_TUNED_SUMMARY_PATH)
print("Notes path:", CLICK_TUNING_NOTES_PATH)

Tuning results path: /Users/khanhhien/Documents/Thesis/Outputs/8_click_tuning_results_v1.csv
Best params path: /Users/khanhhien/Documents/Thesis/Outputs/8_click_best_params_v1.json
Summary path: /Users/khanhhien/Documents/Thesis/Outputs/8_click_tuned_model_summary_v1.csv
Notes path: /Users/khanhhien/Documents/Thesis/Outputs/8_click_tuning_notes_v1.md


## 8.10 Helper functions to extract GridSearchCV results

In [25]:
# GridSearchCV result extraction
def extract_grid_results(
    grid_search,
    candidate_name,
    primary_metric
):
    """
    Convert GridSearchCV cv_results_ into a clean dataframe.
    """

    results = pd.DataFrame(grid_search.cv_results_).copy()

    train_col = f"mean_train_{primary_metric}"
    val_col = f"mean_test_{primary_metric}"

    results["candidate"] = candidate_name

    results["train_val_gap"] = (
        results[train_col]
        - results[val_col]
    )

    return results

In [26]:
# best parameter extraction
def extract_best_model_summary(
    grid_search,
    candidate_name,
    primary_metric
):
    """
    Extract summary of the best GridSearchCV solution.
    """

    best_idx = grid_search.best_index_

    cv_results = pd.DataFrame(grid_search.cv_results_)

    row = cv_results.iloc[best_idx]

    summary = {
        "candidate": candidate_name,
        "best_params": grid_search.best_params_,
        "best_score": grid_search.best_score_,
        "train_score": row[f"mean_train_{primary_metric}"],
        "validation_score": row[f"mean_test_{primary_metric}"],
        "validation_std": row[f"std_test_{primary_metric}"],
        "train_val_gap":
            row[f"mean_train_{primary_metric}"]
            - row[f"mean_test_{primary_metric}"]
    }

    return summary

In [27]:
# Grid Search configuration

N_JOBS = 1

VERBOSE = 3

RETURN_TRAIN_SCORE = True

print("n_jobs =", N_JOBS)
print("verbose =", VERBOSE)
print("return_train_score =", RETURN_TRAIN_SCORE)

n_jobs = 1
verbose = 3
return_train_score = True


## 8.11 Define candidate pipelines

In [56]:
click_candidate_pipelines = {
    "logistic_regression__C_expanded": Pipeline([
        ("preprocessor", click_preprocessor_C),
        ("model", LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=5000,
            random_state=RANDOM_STATE
        ))
    ]),

    "random_forest__C_expanded": Pipeline([
        ("preprocessor", click_preprocessor_C),
        ("model", RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
}

list(click_candidate_pipelines.keys())

['logistic_regression__C_expanded', 'random_forest__C_expanded']

## 8.12 Define hyperparameter grids

In [38]:
click_param_grids = {
    "logistic_regression__C_expanded": {
        "model__C": [0.001, 0.01, 0.1, 1, 10],
        "model__penalty": ["l2"],
        "model__class_weight": [None],
    },


    "random_forest__C_expanded": {
        "model__n_estimators": [200],
        "model__max_depth": [5, 10, 15, 20],
        "model__min_samples_leaf": [5, 10, 25, 50, 100],
        "model__max_features": ["sqrt", 0.5],
        "model__class_weight": [None],
    },
}

In [39]:
# check grid sizes
from sklearn.model_selection import ParameterGrid

for candidate, grid in click_param_grids.items():
    n_combinations = len(list(ParameterGrid(grid)))
    n_total_fits = n_combinations * cv_strategy.get_n_splits()

    print(candidate)
    print("  combinations:", n_combinations)
    print("  total CV fits:", n_total_fits)

logistic_regression__C_expanded
  combinations: 5
  total CV fits: 25
random_forest__C_expanded
  combinations: 40
  total CV fits: 200


## 8.13 Candidate registry

In [40]:
CLICK_PRIMARY_METRIC = "PR_AUC"

CLICK_CANDIDATES = [
    "logistic_regression__C_expanded",
    "random_forest__C_expanded"
]

print("Primary metric:", CLICK_PRIMARY_METRIC)

for candidate in CLICK_CANDIDATES:
    print("-", candidate)

Primary metric: PR_AUC
- logistic_regression__C_expanded
- random_forest__C_expanded


## 8.14 Tune Logistic Regression C

In [53]:
# Logistic Regression GridSearchCV
click_logistic_search = GridSearchCV(
    estimator=click_candidate_pipelines[
        "logistic_regression__C_expanded"
    ],

    param_grid=click_param_grids[
        "logistic_regression__C_expanded"
    ],

    scoring=scoring_click,

    refit=CLICK_PRIMARY_METRIC,

    cv=cv_strategy,

    n_jobs=2,

    verbose=VERBOSE,

    return_train_score=RETURN_TRAIN_SCORE
)

click_logistic_search

GridSearchCV(cv=StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('numeric',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(fill_value=0,
                                                                                                        strategy='constant')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['age_clean',
                                                                          'n_interests',
                                                                          'subject_length',
                                                                          'preheader_length',
                                                                          'prior_email_count...
                                                           random_state=42))]),
             n_jobs=2,
             param_grid={'model__C': [0.001, 0.01, 0.1, 1, 10],
                         'model__class_weight': [None],
                         'model__penalty': ['l2']},
             refit='PR_AUC', return_train_score=True,
             scoring={'PR_AUC': 'average_precision', 'ROC_AUC': 'roc_auc',
                      'accuracy': 'accuracy',
                      'balanced_accuracy': 'balanced_accuracy',
                      'neg_brier': 'neg_brier_score',
                      'neg_log_loss': 'neg_log_loss'},
             verbose=3)

In [42]:
print(click_logistic_search.refit)
print(click_logistic_search.scoring.keys())

PR_AUC
dict_keys(['ROC_AUC', 'PR_AUC', 'accuracy', 'balanced_accuracy', 'neg_brier', 'neg_log_loss'])


In [43]:
# fit Logistic Regression Grid Search
logistic_start_time = time.time()

click_logistic_search.fit(
    X_trainval,
    y_trainval,
    groups=groups_trainval
)

logistic_runtime = time.time() - logistic_start_time

print(f"Runtime: {logistic_runtime:.2f} seconds")

Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV 1/5] END model__C=0.001, model__class_weight=None, model__penalty=l2; PR_AUC: (train=0.339, test=0.478) ROC_AUC: (train=0.958, test=0.963) accuracy: (train=0.999, test=0.999) balanced_accuracy: (train=0.618, test=0.642) neg_brier: (train=-0.001, test=-0.001) neg_log_loss: (train=-0.005, test=-0.006) total time=  14.4s
[CV 2/5] END model__C=0.001, model__class_weight=None, model__penalty=l2; PR_AUC: (train=0.388, test=0.282) ROC_AUC: (train=0.956, test=0.950) accuracy: (train=0.999, test=0.999) balanced_accuracy: (train=0.632, test=0.628) neg_brier: (train=-0.001, test=-0.001) neg_log_loss: (train=-0.006, test=-0.004) total time=  14.6s
[CV 3/5] END model__C=0.001, model__class_weight=None, model__penalty=l2; PR_AUC: (train=0.391, test=0.243) ROC_AUC: (train=0.956, test=0.947) accuracy: (train=0.999, test=0.999) balanced_accuracy: (train=0.639, test=0.573) neg_brier: (train=-0.001, test=-0.001) neg_log_loss: (train=-0.006, 

In [45]:
print("Best parameters:")
print(click_logistic_search.best_params_)

print("\nBest PR-AUC:")
print(click_logistic_search.best_score_)

Best parameters:
{'model__C': 0.1, 'model__class_weight': None, 'model__penalty': 'l2'}

Best PR-AUC:
0.39786997765426635


In [49]:
# extract results
click_logistic_results = extract_grid_results(
    grid_search=click_logistic_search,
    candidate_name="logistic_regression__C_expanded",
    primary_metric=CLICK_PRIMARY_METRIC
)

click_logistic_summary = extract_best_model_summary(
    grid_search=click_logistic_search,
    candidate_name="logistic_regression__C_expanded",
    primary_metric=CLICK_PRIMARY_METRIC
)

click_logistic_summary

{'candidate': 'logistic_regression__C_expanded',
 'best_params': {'model__C': 0.1,
  'model__class_weight': None,
  'model__penalty': 'l2'},
 'best_score': 0.39786997765426635,
 'train_score': 0.4846000644262459,
 'validation_score': 0.39786997765426635,
 'validation_std': 0.09525341174048983,
 'train_val_gap': 0.08673008677197958}

In [50]:
# show key columns only
logistic_display_cols = [
    "candidate",
    "param_model__C",
    "mean_train_PR_AUC",
    "mean_test_PR_AUC",
    "std_test_PR_AUC",
    "train_val_gap",
    "mean_test_ROC_AUC",
    "mean_test_neg_brier",
    "mean_test_neg_log_loss",
    "rank_test_PR_AUC"
]

click_logistic_results[logistic_display_cols].sort_values("rank_test_PR_AUC")

,candidate,param_model__C,mean_train_PR_AUC,mean_test_PR_AUC,std_test_PR_AUC,train_val_gap,mean_test_ROC_AUC,mean_test_neg_brier,mean_test_neg_log_loss,rank_test_PR_AUC
2,logistic_regression__C_expanded,0.100,0.484600,0.397870,0.095253,0.086730,0.955548,-0.000933,-0.004370,1
3,logistic_regression__C_expanded,1.000,0.492048,0.386815,0.091283,0.105233,0.956329,-0.000947,-0.004403,2
4,logistic_regression__C_expanded,10.000,0.492743,0.384144,0.090343,0.108599,0.956425,-0.000952,-0.004411,3
1,logistic_regression__C_expanded,0.010,0.423945,0.373944,0.082582,0.050001,0.950742,-0.000979,-0.004958,4
0,logistic_regression__C_expanded,0.001,0.371996,0.356558,0.089119,0.015438,0.952139,-0.001024,-0.005808,5


In [51]:
# save Logistic Regression tuning
logistic_results_path = OUTPUT_DIR / "8_click_logistic_tuning_results_v1.csv"
logistic_summary_path = OUTPUT_DIR / "8_click_logistic_tuning_summary_v1.json"

click_logistic_results.to_csv(logistic_results_path, index=False)

with open(logistic_summary_path, "w") as f:
    json.dump(click_logistic_summary, f, indent=4, default=str)

print("Saved logistic results to:", logistic_results_path)
print("Saved logistic summary to:", logistic_summary_path)

Saved logistic results to: /Users/khanhhien/Documents/Thesis/Outputs/8_click_logistic_tuning_results_v1.csv
Saved logistic summary to: /Users/khanhhien/Documents/Thesis/Outputs/8_click_logistic_tuning_summary_v1.json


Click Logistic Regression selected `C = 0.1`, using L2 regularisation and no class weighting. This configuration achieved the highest validation PR-AUC among the tested logistic settings.

The train-validation gap is larger than in open prediction, which is expected because click is an extreme rare-event task and PR-AUC is highly sensitive to rare positive cases across grouped campaign folds. However, the result does not suggest uncontrolled overfitting: stronger regularisation (`C = 0.001` and `C = 0.01`) produced smaller gaps but lower validation PR-AUC, while weaker regularisation (`C = 1` and `C = 10`) increased the gap without improving validation PR-AUC.

Therefore, `C = 0.1` is retained as the best balance between predictive performance and regularisation for the click logistic model.

## 8.16 Tune Random Forest C

In [57]:
# random Forest GridSearchCV
click_rf_search = GridSearchCV(
    estimator=click_candidate_pipelines[
        "random_forest__C_expanded"
    ],
    param_grid=click_param_grids[
        "random_forest__C_expanded"
    ],
    scoring=scoring_click,
    refit=CLICK_PRIMARY_METRIC,
    cv=cv_strategy,
    n_jobs=1,
    verbose=VERBOSE,
    return_train_score=RETURN_TRAIN_SCORE
)

print("Refit metric:", click_rf_search.refit)
print("Scoring metrics:", click_rf_search.scoring.keys())

Refit metric: PR_AUC
Scoring metrics: dict_keys(['ROC_AUC', 'PR_AUC', 'accuracy', 'balanced_accuracy', 'neg_brier', 'neg_log_loss'])


In [58]:
# fit Random Forest Grid Search
rf_start_time = time.time()

click_rf_search.fit(
    X_trainval,
    y_trainval,
    groups=groups_trainval
)

rf_runtime = time.time() - rf_start_time

print(f"Runtime: {rf_runtime:.2f} seconds")

Fitting 5 folds for each of 40 candidates, totalling 200 fits
[CV 1/5] END model__class_weight=None, model__max_depth=5, model__max_features=sqrt, model__min_samples_leaf=5, model__n_estimators=200; PR_AUC: (train=0.560, test=0.535) ROC_AUC: (train=0.980, test=0.960) accuracy: (train=0.999, test=0.999) balanced_accuracy: (train=0.566, test=0.541) neg_brier: (train=-0.001, test=-0.001) neg_log_loss: (train=-0.004, test=-0.005) total time=  18.4s
[CV 2/5] END model__class_weight=None, model__max_depth=5, model__max_features=sqrt, model__min_samples_leaf=5, model__n_estimators=200; PR_AUC: (train=0.584, test=0.400) ROC_AUC: (train=0.979, test=0.950) accuracy: (train=0.999, test=0.999) balanced_accuracy: (train=0.584, test=0.562) neg_brier: (train=-0.001, test=-0.001) neg_log_loss: (train=-0.004, test=-0.003) total time=  23.1s
[CV 3/5] END model__class_weight=None, model__max_depth=5, model__max_features=sqrt, model__min_samples_leaf=5, model__n_estimators=200; PR_AUC: (train=0.593, test=

In [59]:
print("Best parameters:")
print(click_rf_search.best_params_)

print("\nBest PR_AUC:")
print(click_rf_search.best_score_)

Best parameters:
{'model__class_weight': None, 'model__max_depth': 15, 'model__max_features': 0.5, 'model__min_samples_leaf': 5, 'model__n_estimators': 200}

Best PR_AUC:
0.5374742666076311


In [60]:
click_rf_results = extract_grid_results(
    grid_search=click_rf_search,
    candidate_name="random_forest__C_expanded",
    primary_metric=CLICK_PRIMARY_METRIC
)

click_rf_summary = extract_best_model_summary(
    grid_search=click_rf_search,
    candidate_name="random_forest__C_expanded",
    primary_metric=CLICK_PRIMARY_METRIC
)

In [61]:
rf_display_cols = [
    "candidate",
    "param_model__n_estimators",
    "param_model__max_depth",
    "param_model__min_samples_leaf",
    "param_model__max_features",
    "mean_train_PR_AUC",
    "mean_test_PR_AUC",
    "std_test_PR_AUC",
    "train_val_gap",
    "mean_test_ROC_AUC",
    "rank_test_PR_AUC"
]

click_rf_results[rf_display_cols].sort_values(
    "rank_test_PR_AUC"
)

,candidate,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features,mean_train_PR_AUC,mean_test_PR_AUC,std_test_PR_AUC,train_val_gap,mean_test_ROC_AUC,rank_test_PR_AUC
25,random_forest__C_expanded,200,15,5,0.5,0.919335,0.537474,0.081883,0.381860,0.938937,1
35,random_forest__C_expanded,200,20,5,0.5,0.929478,0.535326,0.085526,0.394152,0.943612,2
15,random_forest__C_expanded,200,10,5,0.5,0.845040,0.520606,0.090724,0.324434,0.950959,3
36,random_forest__C_expanded,200,20,10,0.5,0.850284,0.514775,0.086063,0.335509,0.950301,4
26,random_forest__C_expanded,200,15,10,0.5,0.844746,0.512371,0.089516,0.332375,0.948288,5
16,random_forest__C_expanded,200,10,10,0.5,0.791665,0.508936,0.091069,0.282729,0.952205,6
30,random_forest__C_expanded,200,20,5,sqrt,0.888477,0.508332,0.079510,0.380145,0.957427,7
20,random_forest__C_expanded,200,15,5,sqrt,0.868591,0.507018,0.075172,0.361573,0.955259,8
10,random_forest__C_expanded,200,10,5,sqrt,0.778921,0.490467,0.086708,0.288453,0.958233,9
21,random_forest__C_expanded,200,15,10,sqrt,0.786878,0.485270,0.084528,0.301608,0.962070,10


In [62]:
# save Random Forest tuning backup
rf_results_path = OUTPUT_DIR / "8_click_random_forest_tuning_results_v1.csv"
rf_summary_path = OUTPUT_DIR / "8_click_random_forest_tuning_summary_v1.json"

click_rf_results.to_csv(
    rf_results_path,
    index=False
)

with open(rf_summary_path, "w") as f:
    json.dump(
        click_rf_summary,
        f,
        indent=4,
        default=str
    )

print("Saved RF results to:", rf_results_path)
print("Saved RF summary to:", rf_summary_path)

Saved RF results to: /Users/khanhhien/Documents/Thesis/Outputs/8_click_random_forest_tuning_results_v1.csv
Saved RF summary to: /Users/khanhhien/Documents/Thesis/Outputs/8_click_random_forest_tuning_summary_v1.json


### Tuning observations

The tuning results show a clear relationship between model complexity and predictive performance. Configurations with deeper trees and smaller terminal nodes generally achieved higher validation PR-AUC values, indicating that click behaviour contains nonlinear patterns that can be exploited by more flexible models. However, these gains were accompanied by substantially larger train-validation gaps, suggesting increased overfitting pressure.

The highest validation PR-AUC was achieved by configurations with max_depth between 15 and 20 and min_samples_leaf equal to 5. As model complexity increased, training PR-AUC rose sharply while validation PR-AUC improved more gradually, leading to large train-validation gaps. Conversely, heavily regularised configurations with larger terminal node sizes produced much smaller gaps but also substantially lower validation PR-AUC values.

Overall, the tuning results suggest that click prediction benefits from nonlinear modelling, but that model complexity must be carefully controlled to balance predictive performance and generalisation.

### Model selection decision

Although GridSearchCV identified a configuration with max_depth = 15, min_samples_leaf = 5, and max_features = 0.5 as the highest-ranked model, the selection was not accepted automatically.

The winning configuration achieved the highest validation PR-AUC but also exhibited a very large train-validation gap, indicating substantial overfitting pressure. While some degree of train-validation divergence is expected in an extreme rare-event problem such as click prediction, the magnitude of the observed gap suggested that the model may be fitting patterns specific to the training folds rather than learning relationships that generalise consistently across campaigns.

A slightly more regularised configuration with max_depth = 10, min_samples_leaf = 5, and max_features = 0.5 was therefore preferred. This model retained most of the predictive improvement achieved through tuning while reducing model complexity and overfitting risk. The resulting validation PR-AUC remained substantially higher than both the logistic regression benchmark and the Stage 6 random forest model, providing a more balanced trade-off between predictive performance and generalisation.

## 8.17 Click tuned candidate comparison

This section compares the tuned click-prediction candidates using cross-validation results only.  
The final chronological test set is still not used.

The purpose is to decide which tuned configurations should be carried forward to the refit/final evaluation stage.

In [63]:
# combine tuned model summaries
click_model_summaries = [
    click_logistic_summary,
    click_rf_summary
]

click_tuned_summary_df = pd.DataFrame(
    click_model_summaries
)

click_tuned_summary_df

,candidate,best_params,best_score,train_score,validation_score,validation_std,train_val_gap
0,logistic_regression__C_expanded,"{'model__C': 0.1, 'model__class_weight': None,...",0.397870,0.484600,0.397870,0.095253,0.08673
1,random_forest__C_expanded,"{'model__class_weight': None, 'model__max_dept...",0.537474,0.919335,0.537474,0.081883,0.38186


In [64]:
# click tuned summary table
click_summary_display = click_tuned_summary_df.copy()

click_summary_display["conservative_score"] = (
    click_summary_display["validation_score"]
    - click_summary_display["validation_std"]
)

click_summary_display = click_summary_display[
    [
        "candidate",
        "best_params",
        "train_score",
        "validation_score",
        "validation_std",
        "conservative_score",
        "train_val_gap"
    ]
].sort_values("validation_score", ascending=False)

click_summary_display

,candidate,best_params,train_score,validation_score,validation_std,conservative_score,train_val_gap
1,random_forest__C_expanded,"{'model__class_weight': None, 'model__max_dept...",0.919335,0.537474,0.081883,0.455591,0.38186
0,logistic_regression__C_expanded,"{'model__C': 0.1, 'model__class_weight': None,...",0.484600,0.397870,0.095253,0.302617,0.08673


In [66]:
# near best random forest alternatives
rf_near_best = click_rf_results[
    [
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "param_model__max_features",
        "mean_train_PR_AUC",
        "mean_test_PR_AUC",
        "std_test_PR_AUC",
        "train_val_gap",
        "mean_test_ROC_AUC",
        "rank_test_PR_AUC"
    ]
].sort_values("rank_test_PR_AUC")

rf_near_best.head(20)

,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features,mean_train_PR_AUC,mean_test_PR_AUC,std_test_PR_AUC,train_val_gap,mean_test_ROC_AUC,rank_test_PR_AUC
25,15,5,0.5,0.919335,0.537474,0.081883,0.381860,0.938937,1
35,20,5,0.5,0.929478,0.535326,0.085526,0.394152,0.943612,2
15,10,5,0.5,0.845040,0.520606,0.090724,0.324434,0.950959,3
36,20,10,0.5,0.850284,0.514775,0.086063,0.335509,0.950301,4
26,15,10,0.5,0.844746,0.512371,0.089516,0.332375,0.948288,5
16,10,10,0.5,0.791665,0.508936,0.091069,0.282729,0.952205,6
30,20,5,sqrt,0.888477,0.508332,0.079510,0.380145,0.957427,7
20,15,5,sqrt,0.868591,0.507018,0.075172,0.361573,0.955259,8
10,10,5,sqrt,0.778921,0.490467,0.086708,0.288453,0.958233,9
21,15,10,sqrt,0.786878,0.485270,0.084528,0.301608,0.962070,10


In [67]:
click_carry_forward_params = {
    "logistic_regression__C_expanded": {
        "model__C": 0.1,
        "model__penalty": "l2",
        "model__class_weight": None,
    },

    "random_forest__C_expanded": {
        "model__n_estimators": 200,
        "model__max_depth": 10,
        "model__min_samples_leaf": 5,
        "model__max_features": "0.5",
        "model__class_weight": None,
    },
}

In [68]:
click_carry_forward_path = OUTPUT_DIR / "8_click_carry_forward_params_v1.json"

with open(click_carry_forward_path, "w") as f:
    json.dump(click_carry_forward_params, f, indent=4, default=str)

print("Saved click carry-forward params to:")
print(click_carry_forward_path)

Saved click carry-forward params to:
/Users/khanhhien/Documents/Thesis/Outputs/8_click_carry_forward_params_v1.json


## 8.18 Stage 6 vs Stage 8 comparison

This section compares the shortlisted candidate models before and after hyperparameter tuning. The objective is not to re-rank models, but to evaluate whether hyperparameter optimisation improved validation performance and increased cross-validation stability.

Three aspects are considered:

* Validation ROC-AUC (predictive performance)
* Validation standard deviation across folds (stability)

This comparison helps quantify the contribution of Stage 8 beyond simple model selection.

In [69]:
stage6_click_comparison = pd.read_csv(
    OUTPUT_DIR / "6_click_comparison_v1.csv"
)

stage6_shortlisted = stage6_click_comparison[
    stage6_click_comparison["candidate_key"].isin([
        "logistic_regression__C_expanded",
        "random_forest__C_expanded"
    ])
].copy()

stage6_shortlisted = stage6_shortlisted[
    [
        "candidate_key",
        "mean_PR_AUC",
        "std_PR_AUC",
        "mean_ROC_AUC",
        "mean_brier",
        "mean_log_loss"
    ]
].rename(columns={
    "candidate_key": "candidate",
    "mean_PR_AUC": "stage6_PR_AUC",
    "std_PR_AUC": "stage6_std_PR_AUC",
    "mean_ROC_AUC": "stage6_ROC_AUC",
    "mean_brier": "stage6_brier",
    "mean_log_loss": "stage6_log_loss"
})

stage8_summary = click_tuned_summary_df[
    [
        "candidate",
        "validation_score",
        "validation_std",
        "train_val_gap"
    ]
].rename(columns={
    "validation_score": "stage8_PR_AUC",
    "validation_std": "stage8_std_PR_AUC",
    "train_val_gap": "stage8_train_val_gap"
})

stage6_vs_stage8 = stage6_shortlisted.merge(
    stage8_summary,
    on="candidate",
    how="left"
)

stage6_vs_stage8["PR_AUC_change"] = (
    stage6_vs_stage8["stage8_PR_AUC"]
    - stage6_vs_stage8["stage6_PR_AUC"]
)

stage6_vs_stage8["std_change"] = (
    stage6_vs_stage8["stage8_std_PR_AUC"]
    - stage6_vs_stage8["stage6_std_PR_AUC"]
)

stage6_vs_stage8.sort_values(
    "stage8_PR_AUC",
    ascending=False
)

,candidate,stage6_PR_AUC,stage6_std_PR_AUC,stage6_ROC_AUC,stage6_brier,stage6_log_loss,stage8_PR_AUC,stage8_std_PR_AUC,stage8_train_val_gap,PR_AUC_change,std_change
0,random_forest__C_expanded,0.508435,0.088197,0.927475,0.000815,0.009183,0.537474,0.081883,0.38186,0.029039,-0.006314
1,logistic_regression__C_expanded,0.386815,0.102058,0.956329,0.000947,0.004403,0.397870,0.095253,0.08673,0.011055,-0.006805


Hyperparameter tuning improved both shortlisted click prediction models.

For Logistic Regression C, validation PR-AUC increased from 0.3868 to 0.3979 (+0.0111), while the cross-validation standard deviation decreased from 0.1021 to 0.0953. This indicates a modest improvement in predictive performance together with slightly more stable behaviour across folds.

For Random Forest C, validation PR-AUC increased from 0.5084 to 0.5375 (+0.0290), representing a substantially larger improvement. Cross-validation variability also decreased slightly, suggesting more consistent performance across folds.

Overall, Stage 7 successfully improved the predictive performance of both shortlisted models. The improvement was particularly pronounced for Random Forest. However, the tuned Random Forest also exhibited a considerably larger train-validation gap, indicating increased overfitting pressure. Therefore, final model selection should consider both predictive performance and generalisation characteristics.

In [70]:
stage6_vs_stage8_path = (
    OUTPUT_DIR /
    "8_click_stage6_vs_stage8_comparison_v1.csv"
)

stage6_vs_stage8.to_csv(
    stage6_vs_stage8_path,
    index=False
)

print(stage6_vs_stage8_path)

/Users/khanhhien/Documents/Thesis/Outputs/8_click_stage6_vs_stage8_comparison_v1.csv


## 8.19 Decision notes

### Logistic Regression C

The tuned click Logistic Regression model selected `C = 0.1`, with L2 regularisation and no class weighting. This setting achieved the highest validation PR-AUC among the tested logistic configurations.

The train-validation gap was larger than in the open-prediction task, which is expected because click prediction is an extreme rare-event problem and PR-AUC is highly sensitive to rare positive cases across campaign folds. However, the result does not indicate uncontrolled overfitting. Stronger regularisation produced lower gaps but also lower validation PR-AUC, while weaker regularisation increased the train-validation gap without improving validation PR-AUC. Therefore, `C = 0.1` was retained as the best balance between regularisation and predictive performance for the click logistic model.

### Random Forest C

The tuned Random Forest achieved the strongest validation PR-AUC among the click candidates. The automatic GridSearchCV winner was `max_depth = 15`, `min_samples_leaf = 5`, `max_features = 0.5`, and `n_estimators = 200`. However, this configuration also produced a very large train-validation gap, indicating substantial overfitting pressure.

Because click prediction is an extreme rare-event task, some train-validation divergence is expected. Nevertheless, the size of the gap suggested that the highest-ranked Random Forest configuration may be fitting campaign-specific or rare positive patterns too strongly. Therefore, the GridSearchCV winner was not accepted automatically.

A more conservative Random Forest configuration was preferred for carry-forward: `max_depth = 10`, `min_samples_leaf = 5`, `max_features = 0.5`, and `n_estimators = 200`. This configuration retained a clear improvement over both the tuned Logistic Regression model and the Stage 6 Random Forest model, while reducing model complexity relative to the highest-ranked RF configuration.

### Final click carry-forward decision

The click models carried forward to the refit and final evaluation stage are:

- Logistic Regression C: `C = 0.1`, L2 penalty, no class weighting.
- Random Forest C: `n_estimators = 200`, `max_depth = 10`, `min_samples_leaf = 5`, `max_features = 0.5`, no class weighting.

These selections reflect a balance between validation PR-AUC, train-validation gap, cross-validation stability, and model complexity. The final chronological test set remains untouched and will only be used in the next stage.

In [71]:
click_tuned_summary_path = (
    OUTPUT_DIR /
    "8_click_tuned_model_summary_v1.csv"
)

click_tuned_summary_df.to_csv(
    click_tuned_summary_path,
    index=False
)

print(click_tuned_summary_path)

/Users/khanhhien/Documents/Thesis/Outputs/8_click_tuned_model_summary_v1.csv
